# 04 — Mechanism-signal decomposition (repeatable)

Decomposes the LLM causal-plausibility feature (`mech_*`, the four axes centrality / population_fit /
effect_ceiling / fitness) into **how much is recoverable from clean public biology** and **what stores it**,
per axis, and tests the within-disease **mechanism-impact** question ("is the known target the right one to
impact THIS disease?") against trial outcome with domain-appropriate clean sources.

Inputs (all built/cached by prior steps): `data/sources/training_dataset_v8_clean_mort.csv`,
target map `ik14_moa_targets_combined_v1.csv`, and the clean-source caches/features
(OT all-channels, DepMap CRISPR, gnomAD constraint, KEGG/OmniPath/GtoPdb/OncoKB/ClinGen, disease pathophysiology).

**The model is unchanged by this analysis** — it is diagnostic (headline AUCs are reported in the
manuscript / `02` / `00`, not restated here).

## 1. Per-axis × per-source recovery matrix (total + within-disease)
How much of each LLM axis is explained (compound-grouped CV R²) by each clean source, and — residualizing on disease-mean — the within-disease *drug-resolution* component.

In [1]:
import subprocess
print(subprocess.run(['python3','scripts/strengthening/compute_axis_recovery.py'],capture_output=True,text=True,cwd='..').stdout)

R^2 (compound-grouped CV) explaining each LLM axis, by data source:

SOURCE                    centralitypopulation_fiteffect_ceiling       fitness
------------------------------------------------------------------------------
OT-association                 0.050         0.030         0.037         0.036
KEGG-pathway                   0.017         0.037         0.024         0.026
Topology(OmniPath)            -0.047        -0.034        -0.018        -0.043
Directional-signed            -0.002        -0.018        -0.003        -0.009
DepMap-dependency             -0.001        -0.014         0.004        -0.003
ATC-drugclass                  0.067         0.003         0.083         0.048
GtoPdb-physiology             -0.003        -0.001        -0.002        -0.002
OncoKB-driver                  0.014        -0.003         0.009         0.007
gnomAD-constraint              0.039        -0.032         0.048         0.014
Disease-pathophys              0.069         0.062         0.0

**Reads (see `notes/llm_axis_recovery_table.md`):**
- Total recovery is DISEASE-LEVEL: centrality ~0.22, effect_ceiling ~0.23, fitness ~0.19, population_fit ~0.12.
  Biggest stores: Disease-pathophysiology, ATC drug-class, gnomAD constraint, OT genetics.
- Within-disease (residualized) the drug-side sources recover ~0% of the LLM *number* — but that is the wrong
  target; the right test is OUTCOME (below).

## 2. Mechanism-impact score: does the known target predict OUTCOME within a disease?
Domain-appropriate clean routing — DepMap selective dependency (oncology), OT human genetics (else).

In [2]:
print(subprocess.run(['python3','scripts/strengthening/build_mechanism_impact.py'],capture_output=True,text=True,cwd='..').stdout)
print(subprocess.run(['python3','scripts/strengthening/domain_conditional_impact_test.py'],capture_output=True,text=True,cwd='..').stdout)

wrote mechanism_impact_v1.csv (2336 rows; 14% routed to DepMap)



DOMAIN-CONDITIONAL mechanism-impact, pooled within-disease AUC: 0.627  (naive combined was 0.582; per-domain 0.6-0.74; LLM 0.724)
mean per-disease AUC (n-weighted): 0.642 across 47 diseases
drug-bootstrap 95% CI [0.551, 0.708]
  oncology: 0.710 (n=197)
  non-onc: 0.591 (n=640)



**Reads:** within-disease efficacy AUC ~0.627 pooled (oncology **0.71** via DepMap, matching the LLM's
within-disease 0.724). The 4 LLM axes (0.84–0.96 correlated) collapse to ONE recoverable signal — *is the
target a core causal driver of this disease* — captured cleanly by mechanism-impact.

## 3. Oncology vs non-oncology: why the asymmetry
Non-oncology within-disease efficacy AUC by OT channel (clean vs outcome-adjacent).

In [3]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score
df=pd.read_csv('../data/sources/training_dataset_v8_clean_mort.csv',low_memory=False); df['IK14']=df['feature_IK'].astype(str).str[:14]
ot=pd.read_csv('../data/sources/ot_channel_features_v1.csv')
e=df[(df.Corrected_Outcome.isin(['PASS','FAIL_EFFICACY','FAIL_BOTH']))&(df.disease_is_oncology!=1)].copy()
e['y']=e.Corrected_Outcome.isin(['FAIL_EFFICACY','FAIL_BOTH']).astype(int)
chans={'genetic (CLEAN)':['ot_genetic_association_max','ot_genetic_literature_max'],'animal-model (CLEAN)':['ot_animal_model_max'],
       'pathway (CLEAN)':['ot_affected_pathway_max'],'clinical (OUTCOME-ADJ)':['ot_clinical_max'],'literature (OUTCOME-ADJ)':['ot_literature_max']}
allc=[c for v in chans.values() for c in v if c in ot.columns]
e=e.merge(ot[['IK14','Disease']+allc].drop_duplicates(['IK14','Disease']),on=['IK14','Disease'],how='left')
for c in allc: e[c]=e[c].fillna(0.0)
big=e.groupby('Disease').filter(lambda g:g.y.nunique()==2 and len(g)>=8)
for name,cols in chans.items():
    cc=[c for c in cols if c in big.columns]
    if not cc: continue
    s=big.copy(); s['v']=s[cc].max(axis=1); s['r']=s.v-s.groupby('Disease').v.transform('mean')
    print(f'{name:26s} within-disease AUC = {roc_auc_score(s.y,-s.r):.3f}')

genetic (CLEAN)            within-disease AUC = 0.614
animal-model (CLEAN)       within-disease AUC = 0.548
pathway (CLEAN)            within-disease AUC = 0.500
clinical (OUTCOME-ADJ)     within-disease AUC = 0.683
literature (OUTCOME-ADJ)   within-disease AUC = 0.598


**Reads:** clean non-oncology ceiling ~0.61 (genetics); the better signal (clinical 0.68) is outcome-adjacent.
**Cause** (characterized, not "missing database"): physiological-therapeutic targets (ACE, dopamine receptors,
SGLT2) are not disease-risk genes and lack a CRISPR-equivalent clean functional readout — their disease-impact
is pharmacological/clinical, the outcome-adjacent layer the LLM memorized.

## Conclusion
The LLM mechanism judgment decomposes into:
1. **A clean, recoverable mechanism-impact signal** — oncology 0.71 (functional genomics), non-oncology 0.61
   (genetics), + ~20% disease-level baseline (pathophysiology/ATC/constraint). Auditable, no LLM.
2. **An outcome-adjacent / irreducible remainder** — population/effect-size + clinical memory — excluded from a
   defensible model.

Full tracker + numbers: `notes/llm_axis_recovery_table.md`.